# MIH Influencer ML Training Pipeline (Kaggle Version)
This notebook processes the massive Instagram dataset and retrains the MIH models.

## 1. Fast Data Aggregation

In [ ]:
import os, json, joblib
import pandas as pd
import numpy as np
from concurrent.futures import ProcessPoolExecutor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Paths
JSON_DIR = '/kaggle/input/mih-influencer-dataset/json_files-017/json'
BASE_CSV = '/kaggle/input/mih-influencer-dataset/top_1000_instagrammers.csv'

def extract(f):
    try:
        with open(os.path.join(JSON_DIR, f), 'r') as file:
            d = json.load(file)
        return {'id': d['owner']['id'], 'l': d['edge_media_preview_like']['count'], 'c': d['edge_media_to_comment']['count']}
    except: return None

files = [f for f in os.listdir(JSON_DIR) if f.endswith('.json')]
with ProcessPoolExecutor() as exe:
    results = list(tqdm(exe.map(extract, files), total=len(files)))

agg = pd.DataFrame([r for r in results if r]).groupby('id').agg({'l': 'mean', 'c': 'mean'}).reset_index()
agg.to_csv('aggregated.csv', index=False)